# What you cannot get back

Once two integers have become the same bit, no later function of that bit tells them apart. Once the bit itself is collapsed to a point, no function of that point returns both `False` and `True`. And a commuting square — even a honest one — is not a decision in the coarser world: the target still has to produce a terminal.

This notebook shows those three refusals.

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_mur_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "mur" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "mur"
elif (p / "mur" / "code").is_dir():
    root = p / "mur"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))
repo_root = root.parents[2] if len(root.parents) >= 3 else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from sympy import Matrix, Rational
from coda_mur_core import (
    DescentSquareCertificate,
    LossLedger,
    MorphismKind,
    TerminalRegime,
    compute_transport_quotient_obstruction,
    evaluate_transport_authority,
)

## Collisions travel

$2$ and $4$ already look the same after parity. A commuting transport still reports one coarser value, not two. A later function of that value — here, a tiny tag — cannot split them.

In [2]:
def look(n):
    return n % 2

def states(n):
    return n % 2

def observations(bit):
    return bit

def commutes(n):
    return observations(look(n)) == look(states(n))

assert commutes(2) and commutes(4)
assert look(2) == look(4)
assert look(states(2)) == look(states(4))

def postprocess(bit):
    return f"tagged_{bit}"

assert postprocess(look(2)) == postprocess(look(4))
print("2 and 4 collide at", look(2))
print("after a later tag they still collide at", postprocess(look(2)))

2 and 4 collide at 0
after a later tag they still collide at tagged_0


## A collapsed point does not return both bits

Send everything to a single point `()`. A recovery map would be a function of `()` that equals every bit. There is no such function: one point yields one bit, not both. That is the whole obstruction.

In [3]:
def exists_recover():
    candidates = [lambda: False, lambda: True]
    for recover in candidates:
        if all(recover() is b for b in (False, True)):
            return True
    return False

print("a recover map that hits both bits exists?", exists_recover())
assert exists_recover() is False

empty_ledger = LossLedger.empty()
unpriced = DescentSquareCertificate(
    source_regime="integers",
    target_regime="parity",
    morphism_name="mod2_silent",
    morphism_kind=MorphismKind.QUOTIENT,
    chi_certified=True,
    commutes=True,
    loss_ledger=empty_ledger,
    carrier=("2", "3", "4"),
    state_map=lambda x: str(int(x) % 2),
    observation_map=lambda x: str(int(x) % 2),
    source_look=lambda x: x,
    target_look=lambda x: x,
)
target = TerminalRegime(
    regime_id="parity",
    terminalize=lambda y: f"bit_{y}",
    is_auth=lambda t: True,
)
silent = evaluate_transport_authority("4", target, "0", unpriced)
print("quotient with empty ledger:", silent.inhabited, silent.refusal_code)
assert silent.inhabited is False
assert silent.refusal_code == "R_unpriced_loss"

a recover map that hits both bits exists? False
quotient with empty ledger: False R_unpriced_loss


## A commuting square is not target standing

Keep a valid exact certificate. Ask a target that never produces a terminal. The square commutes; standing is still empty. That is `R_noterminal`.

A leftover of a linear cochain problem is a different refusal. Licensed repairs are multiples of $(1,-1)$. The vector $(1,1)$ is leftover $2$, and the calculator reports `R_obstruction`. Measuring leftover is not a construction of standing from $\chi_F$.

In [4]:
from dataclasses import replace
exact = DescentSquareCertificate(
    source_regime="integers",
    target_regime="integers_again",
    morphism_name="identity",
    morphism_kind=MorphismKind.EXACT,
    chi_certified=True,
    commutes=True,
    carrier=("2", "3", "4"),
    state_map=lambda x: x,
    observation_map=lambda x: x,
    source_look=lambda x: x,
    target_look=lambda x: x,
)
mute = TerminalRegime(
    regime_id="integers_again",
    terminalize=lambda y: None,
    is_auth=lambda t: True,
)
no_term = evaluate_transport_authority("4", mute, "4", exact)
print("commuting, but no terminal:", no_term.inhabited, no_term.refusal_code)
assert no_term.inhabited is False
assert no_term.refusal_code == "R_noterminal"

d0 = Matrix([[1], [-1]])
on_line = compute_transport_quotient_obstruction(d0, Matrix([[2], [-2]]))
off_line = compute_transport_quotient_obstruction(d0, Matrix([[1], [1]]))
print("leftover on the line:", on_line.distance)
print("leftover off the line:", off_line.distance)
assert on_line.distance == Rational(0)
assert off_line.distance == Rational(2)

live = TerminalRegime(
    regime_id="integers_again",
    terminalize=lambda y: f"term_{y}",
    is_auth=lambda t: True,
)
blocked = evaluate_transport_authority(
    "4", live, "4", replace(exact, defect_vector=off_line.cochain, repair_map=d0),
    quotient_certificate=off_line
)
print("positive leftover:", blocked.inhabited, blocked.refusal_code, blocked.residual_mass)
assert blocked.inhabited is False
assert blocked.refusal_code == "R_obstruction"

commuting, but no terminal: False R_noterminal
leftover on the line: 0
leftover off the line: 2
positive leftover: False R_obstruction 2
